
# Kuhn Poker DREAM Baseline Experiment

This notebook implements a DREAM-style baseline for OpenSpiel Kuhn poker. It is designed to match the thesis analysis format used for the Deep CFR and ESCHER baseline notebooks: fixed-budget multi-seed training, exploitability as the primary metric, policy-value error as a secondary metric, and consistent CSV/JSON/PNG exports.

The implementation is an OpenSpiel/PyTorch adaptation of DREAM rather than a direct copy of the original PokerRL implementation. It preserves the main DREAM ideas used in this thesis comparison: outcome-sampling regret updates, epsilon-mixed sampling for the traverser, learned advantage baselines, and a learned average-policy network for evaluation.


In [ ]:

from __future__ import annotations

import collections
import csv
import json
import math
import os
import random
import time
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Deque, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

try:
    import pyspiel
    from open_spiel.python import policy
    from open_spiel.python.algorithms import expected_game_score
    from open_spiel.python.algorithms import exploitability
except Exception as exc:  # pragma: no cover - OpenSpiel may not be installed in all environments.
    pyspiel = None
    policy = None
    expected_game_score = None
    exploitability = None
    print("OpenSpiel import failed. Install open_spiel before running the experiment.")
    print(exc)


In [ ]:

# -----------------------------
# Reproducibility and constants
# -----------------------------

KUHN_GAME_VALUE_P0 = -1.0 / 18.0
KUHN_AVERAGE_POLICY_VALUE_TARGET = KUHN_GAME_VALUE_P0
EXPLOITABILITY_THRESHOLD = 0.05

DEFAULT_SEEDS_5 = [1234, 2025, 31415, 27182, 16180]
THESIS_SEEDS_10 = [1234, 2025, 31415, 27182, 16180, 4242, 8675309, 7, 99, 1001]
SMOKE_TEST_SEEDS = [1234, 2025]


def set_seed(seed: int) -> None:
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_mean(xs: Sequence[float]) -> float:
    arr = np.asarray(xs, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    return float(arr.mean()) if len(arr) else float("nan")


def safe_std(xs: Sequence[float]) -> float:
    arr = np.asarray(xs, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    return float(arr.std(ddof=1)) if len(arr) > 1 else 0.0


def standard_error(xs: Sequence[float]) -> float:
    arr = np.asarray(xs, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    return float(arr.std(ddof=1) / np.sqrt(len(arr))) if len(arr) > 1 else 0.0


def compute_auc(x: Sequence[float], y: Sequence[float]) -> float:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2:
        return float("nan")
    span = x[-1] - x[0]
    if span <= 0:
        return float("nan")
    return float(np.trapz(y, x) / span)


In [ ]:

# -----------------------------
# Memory records and replay buffers
# -----------------------------

DreamAdvantageMemory = collections.namedtuple(
    "DreamAdvantageMemory", "info_state iteration advantage weight"
)
DreamStrategyMemory = collections.namedtuple(
    "DreamStrategyMemory", "info_state iteration strategy_action_probs weight"
)
BaselineTransition = collections.namedtuple(
    "BaselineTransition",
    "info_state action reward next_info_state next_legal_actions next_player done",
)


class ReservoirBuffer:
    """Reservoir sampling buffer with bounded memory."""

    def __init__(self, capacity: int):
        self._capacity = int(capacity)
        self._data: List = []
        self._add_calls = 0

    def __len__(self) -> int:
        return len(self._data)

    @property
    def add_calls(self) -> int:
        return int(self._add_calls)

    def add(self, element) -> None:
        self._add_calls += 1
        if len(self._data) < self._capacity:
            self._data.append(element)
        else:
            idx = random.randrange(self._add_calls)
            if idx < self._capacity:
                self._data[idx] = element

    def sample(self, num_samples: int):
        if num_samples > len(self._data):
            raise ValueError("Cannot sample more elements than are currently stored.")
        return random.sample(self._data, num_samples)

    def sample_up_to(self, num_samples: int):
        if not self._data:
            return []
        return random.sample(self._data, min(num_samples, len(self._data)))


class CircularReplay:
    """FIFO replay buffer for baseline learning."""

    def __init__(self, capacity: int):
        self._capacity = int(capacity)
        self._data: Deque = collections.deque(maxlen=self._capacity)
        self._add_calls = 0

    def __len__(self) -> int:
        return len(self._data)

    @property
    def add_calls(self) -> int:
        return int(self._add_calls)

    def add(self, element) -> None:
        self._add_calls += 1
        self._data.append(element)

    def sample(self, num_samples: int):
        if num_samples > len(self._data):
            raise ValueError("Cannot sample more elements than are currently stored.")
        return random.sample(list(self._data), num_samples)

    def sample_up_to(self, num_samples: int):
        data = list(self._data)
        if not data:
            return []
        return random.sample(data, min(num_samples, len(data)))


In [ ]:

# -----------------------------
# Neural network helpers
# -----------------------------

class SonnetLinear(nn.Module):
    """Linear layer using Xavier weight initialisation and zero bias."""

    def __init__(self, in_size: int, out_size: int):
        super().__init__()
        self.layer = nn.Linear(in_size, out_size)
        nn.init.xavier_uniform_(self.layer.weight)
        nn.init.zeros_(self.layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layer(x)


class MLP(nn.Module):
    """Small MLP used for advantage, baseline, and average-policy networks."""

    def __init__(self, input_size: int, hidden_sizes: Sequence[int], output_size: int):
        super().__init__()
        sizes = [int(input_size)] + list(map(int, hidden_sizes)) + [int(output_size)]
        layers: List[nn.Module] = []
        for i in range(len(sizes) - 1):
            layers.append(SonnetLinear(sizes[i], sizes[i + 1]))
            if i < len(sizes) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def grad_norm(parameters: Iterable[torch.nn.Parameter]) -> float:
    total = 0.0
    for p in parameters:
        if p.grad is not None:
            total += float(p.grad.detach().data.norm(2).item() ** 2)
    return float(math.sqrt(total))


In [ ]:

# -----------------------------
# DREAM-style OpenSpiel solver
# -----------------------------

class DREAMSolver(policy.Policy if policy is not None else object):
    """OpenSpiel/PyTorch DREAM-style solver for sequential imperfect-information games.

    This is a thesis-oriented implementation designed to make DREAM comparable with the
    Deep CFR and ESCHER notebooks used in this project. It implements outcome-sampling
    regret updates with learned advantage baselines and a learned average-policy network.

    Important caveat: the official DREAM codebase is implemented in PokerRL and supports
    Single-style averaging. This OpenSpiel adaptation instead exposes an average-policy
    network so that exact exploitability can be computed with the same OpenSpiel tooling
    used elsewhere in the thesis.
    """

    def __init__(
        self,
        game,
        policy_network_layers: Sequence[int] = (32, 32),
        advantage_network_layers: Sequence[int] = (32, 32),
        baseline_network_layers: Optional[Sequence[int]] = None,
        num_iterations: int = 500,
        num_traversals: int = 320,
        learning_rate: float = 3e-3,
        batch_size_advantage: int = 1024,
        batch_size_strategy: int = 1024,
        batch_size_baseline: Optional[int] = None,
        advantage_memory_capacity: int = int(1e6),
        strategy_memory_capacity: int = int(1e6),
        baseline_memory_capacity: int = int(1e6),
        epsilon: float = 0.06,
        advantage_network_train_steps: int = 100,
        policy_network_train_steps: int = 200,
        baseline_network_train_steps: int = 100,
        policy_network_train_every: int = 25,
        compute_exploitability: bool = True,
        seed: Optional[int] = None,
    ):
        if policy is not None:
            super().__init__(game, list(range(game.num_players())))
        if pyspiel is not None and game.get_type().dynamics == pyspiel.GameType.Dynamics.SIMULTANEOUS:
            raise ValueError("DREAMSolver only supports sequential games in this notebook.")

        self._game = game
        self._num_players = int(game.num_players())
        self._num_actions = int(game.num_distinct_actions())
        self._info_state_size = int(game.information_state_tensor_shape()[0])

        self._policy_network_layers = tuple(policy_network_layers)
        self._advantage_network_layers = tuple(advantage_network_layers)
        self._baseline_network_layers = tuple(baseline_network_layers or advantage_network_layers)

        self._num_iterations = int(num_iterations)
        self._num_traversals = int(num_traversals)
        self._learning_rate = float(learning_rate)
        self._epsilon = float(epsilon)
        self._batch_size_advantage = int(batch_size_advantage)
        self._batch_size_strategy = int(batch_size_strategy)
        self._batch_size_baseline = int(batch_size_baseline or batch_size_advantage)
        self._advantage_memory_capacity = int(advantage_memory_capacity)
        self._strategy_memory_capacity = int(strategy_memory_capacity)
        self._baseline_memory_capacity = int(baseline_memory_capacity)
        self._advantage_network_train_steps = int(advantage_network_train_steps)
        self._policy_network_train_steps = int(policy_network_train_steps)
        self._baseline_network_train_steps = int(baseline_network_train_steps)
        self._policy_network_train_every = int(policy_network_train_every)
        self._compute_exploitability = bool(compute_exploitability)

        if seed is not None:
            set_seed(seed)

        self._policy_network = MLP(self._info_state_size, self._policy_network_layers, self._num_actions)
        self._policy_softmax = nn.Softmax(dim=-1)
        self._optimizer_policy = torch.optim.Adam(self._policy_network.parameters(), lr=self._learning_rate)

        self._advantage_networks = [
            MLP(self._info_state_size, self._advantage_network_layers, self._num_actions)
            for _ in range(self._num_players)
        ]
        self._baseline_networks = [
            MLP(self._info_state_size, self._baseline_network_layers, self._num_actions)
            for _ in range(self._num_players)
        ]
        self._optimizer_advantages = [
            torch.optim.Adam(net.parameters(), lr=self._learning_rate) for net in self._advantage_networks
        ]
        self._optimizer_baselines = [
            torch.optim.Adam(net.parameters(), lr=self._learning_rate) for net in self._baseline_networks
        ]

        self._advantage_memories = [ReservoirBuffer(self._advantage_memory_capacity) for _ in range(self._num_players)]
        self._strategy_memories = ReservoirBuffer(self._strategy_memory_capacity)
        self._baseline_replays = [CircularReplay(self._baseline_memory_capacity) for _ in range(self._num_players)]

        self._loss_mse = nn.MSELoss(reduction="mean")
        self._iteration = 0
        self._nodes_touched = 0
        self._last_advantage_loss = [np.nan] * self._num_players
        self._last_baseline_loss = [np.nan] * self._num_players
        self._last_policy_loss = np.nan
        self._last_advantage_grad_norm = [np.nan] * self._num_players
        self._last_baseline_grad_norm = [np.nan] * self._num_players
        self._last_policy_grad_norm = np.nan

    # ---- Public OpenSpiel policy interface ----
    def action_probabilities(self, state, player_id=None) -> Dict[int, float]:
        del player_id
        cur_player = int(state.current_player())
        legal_actions = list(state.legal_actions(cur_player))
        if not legal_actions:
            return {}
        info_state = np.asarray(state.information_state_tensor(cur_player), dtype=np.float32)[None, :]
        with torch.no_grad():
            logits = self._policy_network(torch.from_numpy(info_state))[0].cpu().numpy()
        legal_logits = np.asarray([logits[a] for a in legal_actions], dtype=np.float64)
        legal_logits -= float(np.max(legal_logits))
        exp_logits = np.exp(legal_logits)
        probs = exp_logits / float(np.sum(exp_logits))
        return {int(a): float(p) for a, p in zip(legal_actions, probs)}

    # ---- Main training loop ----
    def solve(self):
        start_time = time.perf_counter()
        curves: List[Dict] = []

        # In the official DREAM repository an iteration denotes a sequential update for both players.
        for iteration in range(1, self._num_iterations + 1):
            self._iteration = int(iteration)
            for traverser in range(self._num_players):
                for _ in range(self._num_traversals):
                    state = self._game.new_initial_state()
                    self._traverse_outcome_sampling(state, traverser, pi_reach=1.0, sigma_reach=1.0)

                self._last_advantage_loss[traverser] = self._learn_advantage_network(traverser)
                self._last_baseline_loss[traverser] = self._learn_baseline_network(traverser)

            train_policy_now = (iteration % self._policy_network_train_every == 0) or (iteration == self._num_iterations)
            if train_policy_now:
                self._last_policy_loss = self._learn_strategy_network()
                curves.append(self._checkpoint_metrics(start_time))

        return pd.DataFrame(curves)

    # ---- Traversal ----
    def _advance_through_chance(self, state):
        while state.is_chance_node():
            outcomes, probs = zip(*state.chance_outcomes())
            action = int(np.random.choice(outcomes, p=probs))
            state = state.child(action)
        return state

    def _traverse_outcome_sampling(self, state, traverser: int, pi_reach: float, sigma_reach: float) -> float:
        self._nodes_touched += 1

        if state.is_terminal():
            return float(state.returns()[traverser])

        if state.is_chance_node():
            outcomes, probs = zip(*state.chance_outcomes())
            action = int(np.random.choice(outcomes, p=probs))
            return self._traverse_outcome_sampling(state.child(action), traverser, pi_reach, sigma_reach)

        current_player = int(state.current_player())
        legal_actions = list(state.legal_actions(current_player))
        info_state = np.asarray(state.information_state_tensor(current_player), dtype=np.float32)

        pi = self._regret_matching_policy(info_state, legal_actions, current_player)
        if current_player == traverser:
            sigma = self._epsilon_mixed_sampling_policy(pi, legal_actions)
        else:
            sigma = pi.copy()

        # Numerical cleanup for sampling.
        sample_probs = np.zeros(self._num_actions, dtype=np.float64)
        for a in legal_actions:
            sample_probs[a] = max(float(sigma[a]), 0.0)
        total = float(np.sum(sample_probs))
        if total <= 0:
            for a in legal_actions:
                sample_probs[a] = 1.0 / len(legal_actions)
        else:
            sample_probs /= total

        action = int(np.random.choice(self._num_actions, p=sample_probs))
        reach_ratio = float(pi_reach / max(sigma_reach, 1e-12))

        # Store average-policy target. The weight carries the correction caused by traverser exploration.
        self._strategy_memories.add(DreamStrategyMemory(info_state, self._iteration, pi.tolist(), reach_ratio))

        next_state = self._advance_through_chance(state.child(action))
        done = next_state.is_terminal()
        reward = float(next_state.returns()[traverser]) if done else 0.0

        if done:
            next_info = np.zeros_like(info_state, dtype=np.float32)
            next_legal_actions: List[int] = []
            next_player = -1
        else:
            next_player = int(next_state.current_player())
            next_info = np.asarray(next_state.information_state_tensor(next_player), dtype=np.float32)
            next_legal_actions = list(next_state.legal_actions(next_player))

        self._baseline_replays[traverser].add(
            BaselineTransition(info_state, action, reward, next_info, next_legal_actions, next_player, done)
        )

        if done:
            child_value = reward
        else:
            if current_player == traverser:
                pi_reach_next = pi_reach * float(pi[action])
                sigma_reach_next = sigma_reach * float(sigma[action])
            else:
                pi_reach_next = pi_reach
                sigma_reach_next = sigma_reach
            child_value = self._traverse_outcome_sampling(next_state, traverser, pi_reach_next, sigma_reach_next)

        baseline_values = self._baseline_values(traverser, info_state)
        sampled_prob = float(sigma[action] if current_player == traverser else pi[action])
        sampled_prob = max(sampled_prob, 1e-12)

        action_values = baseline_values.copy()
        action_values[action] = baseline_values[action] + (child_value - baseline_values[action]) / sampled_prob

        state_value = 0.0
        for a in legal_actions:
            state_value += float(pi[a]) * float(action_values[a])

        if current_player == traverser:
            advantage = np.zeros(self._num_actions, dtype=np.float32)
            for a in legal_actions:
                advantage[a] = float(action_values[a] - state_value)
            self._advantage_memories[traverser].add(
                DreamAdvantageMemory(info_state, self._iteration, advantage.tolist(), reach_ratio)
            )

        return float(state_value)

    # ---- Policy and value helpers ----
    def _regret_matching_policy(self, info_state: np.ndarray, legal_actions: Sequence[int], player: int) -> np.ndarray:
        with torch.no_grad():
            x = torch.from_numpy(info_state.astype(np.float32)[None, :])
            raw_advantages = self._advantage_networks[player](x)[0].cpu().numpy()
        positive = np.maximum(raw_advantages, 0.0)
        denom = float(np.sum(positive[list(legal_actions)]))
        pi = np.zeros(self._num_actions, dtype=np.float32)
        if denom > 0:
            for a in legal_actions:
                pi[a] = positive[a] / denom
        else:
            # Standard regret-matching fallback: uniform over legal actions.
            for a in legal_actions:
                pi[a] = 1.0 / float(len(legal_actions))
        return pi

    def _epsilon_mixed_sampling_policy(self, pi: np.ndarray, legal_actions: Sequence[int]) -> np.ndarray:
        sigma = np.zeros(self._num_actions, dtype=np.float32)
        if not legal_actions:
            return sigma
        uniform = 1.0 / float(len(legal_actions))
        for a in legal_actions:
            sigma[a] = (1.0 - self._epsilon) * float(pi[a]) + self._epsilon * uniform
        sigma_sum = float(np.sum(sigma))
        if sigma_sum > 0:
            sigma /= sigma_sum
        return sigma

    def _baseline_values(self, traverser: int, info_state: np.ndarray) -> np.ndarray:
        with torch.no_grad():
            x = torch.from_numpy(info_state.astype(np.float32)[None, :])
            values = self._baseline_networks[traverser](x)[0].cpu().numpy()
        return values.astype(np.float32)

    # ---- Learning updates ----
    def _learn_advantage_network(self, player: int) -> float:
        if len(self._advantage_memories[player]) < self._batch_size_advantage:
            return float("nan")
        net = self._advantage_networks[player]
        opt = self._optimizer_advantages[player]
        last_loss = float("nan")
        for _ in range(self._advantage_network_train_steps):
            samples = self._advantage_memories[player].sample(self._batch_size_advantage)
            info_states = np.asarray([s.info_state for s in samples], dtype=np.float32)
            targets = np.asarray([s.advantage for s in samples], dtype=np.float32)
            iterations = np.asarray([s.iteration for s in samples], dtype=np.float32).reshape(-1, 1)
            imp_weights = np.asarray([s.weight for s in samples], dtype=np.float32).reshape(-1, 1)
            imp_weights = np.clip(imp_weights, 0.0, 100.0)
            mult = np.sqrt(np.maximum(iterations, 1.0) * np.maximum(imp_weights, 1e-8)).astype(np.float32)
            x = torch.from_numpy(info_states)
            y = torch.from_numpy(targets)
            m = torch.from_numpy(mult)
            opt.zero_grad()
            out = net(x)
            loss = self._loss_mse(m * out, m * y)
            loss.backward()
            self._last_advantage_grad_norm[player] = grad_norm(net.parameters())
            opt.step()
            last_loss = float(loss.detach().cpu().item())
        return last_loss

    def _learn_baseline_network(self, traverser: int) -> float:
        if len(self._baseline_replays[traverser]) < self._batch_size_baseline:
            return float("nan")
        net = self._baseline_networks[traverser]
        opt = self._optimizer_baselines[traverser]
        last_loss = float("nan")
        for _ in range(self._baseline_network_train_steps):
            samples = self._baseline_replays[traverser].sample(self._batch_size_baseline)
            preds: List[torch.Tensor] = []
            targets: List[torch.Tensor] = []
            for tr in samples:
                s = torch.from_numpy(np.asarray(tr.info_state, dtype=np.float32)[None, :])
                q_values = net(s)[0]
                preds.append(q_values[int(tr.action)])
                if bool(tr.done):
                    targets.append(torch.tensor(float(tr.reward), dtype=torch.float32))
                    continue
                next_info = np.asarray(tr.next_info_state, dtype=np.float32)
                next_legal = list(tr.next_legal_actions)
                next_player = int(tr.next_player)
                pi_next = self._regret_matching_policy(next_info, next_legal, next_player)
                with torch.no_grad():
                    q_next = net(torch.from_numpy(next_info[None, :]))[0].cpu().numpy()
                exp_next = sum(float(pi_next[a]) * float(q_next[a]) for a in next_legal)
                targets.append(torch.tensor(float(tr.reward) + exp_next, dtype=torch.float32))
            pred_t = torch.stack(preds)
            target_t = torch.stack(targets)
            opt.zero_grad()
            loss = self._loss_mse(pred_t, target_t)
            loss.backward()
            self._last_baseline_grad_norm[traverser] = grad_norm(net.parameters())
            opt.step()
            last_loss = float(loss.detach().cpu().item())
        return last_loss

    def _learn_strategy_network(self) -> float:
        if len(self._strategy_memories) < self._batch_size_strategy:
            return float("nan")
        last_loss = float("nan")
        for _ in range(self._policy_network_train_steps):
            samples = self._strategy_memories.sample(self._batch_size_strategy)
            info_states = np.asarray([s.info_state for s in samples], dtype=np.float32)
            strategies = np.asarray([s.strategy_action_probs for s in samples], dtype=np.float32)
            iterations = np.asarray([s.iteration for s in samples], dtype=np.float32).reshape(-1, 1)
            imp_weights = np.asarray([s.weight for s in samples], dtype=np.float32).reshape(-1, 1)
            imp_weights = np.clip(imp_weights, 0.0, 100.0)
            mult = np.sqrt(np.maximum(iterations, 1.0) * np.maximum(imp_weights, 1e-8)).astype(np.float32)
            x = torch.from_numpy(info_states)
            y = torch.from_numpy(strategies)
            m = torch.from_numpy(mult)
            self._optimizer_policy.zero_grad()
            logits = self._policy_network(x)
            probs = self._policy_softmax(logits)
            loss = self._loss_mse(m * probs, m * y)
            loss.backward()
            self._last_policy_grad_norm = grad_norm(self._policy_network.parameters())
            self._optimizer_policy.step()
            last_loss = float(loss.detach().cpu().item())
        return last_loss

    # ---- Diagnostics ----
    def _checkpoint_metrics(self, start_time: float) -> Dict:
        tab_policy = policy.tabular_policy_from_callable(self._game, self.action_probabilities)
        policy_value = float(expected_game_score.policy_value(self._game.new_initial_state(), [tab_policy] * self._num_players)[0])
        average_policy_value = policy_value
        if self._compute_exploitability:
            nash_conv = float(exploitability.nash_conv(self._game, tab_policy))
            expl = nash_conv / 2.0
        else:
            nash_conv = float("nan")
            expl = float("nan")
        policy_diag = self._policy_network_diagnostics()
        advantage_diag = self._advantage_target_summary()
        baseline_diag = self._baseline_replay_summary()
        row = {
            "iteration": int(self._iteration),
            "nodes_touched": int(self._nodes_touched),
            "wall_clock_seconds": float(time.perf_counter() - start_time),
            "nash_conv": nash_conv,
            "exploitability": expl,
            "policy_value_player_0": policy_value,
            "average_policy_value": average_policy_value,
            "policy_value_error": abs(policy_value - KUHN_GAME_VALUE_P0),
            "average_policy_value_error": abs(average_policy_value - KUHN_AVERAGE_POLICY_VALUE_TARGET),
            "policy_loss": float(self._last_policy_loss),
            "advantage_loss_player_0": float(self._last_advantage_loss[0]),
            "advantage_loss_player_1": float(self._last_advantage_loss[1]),
            "baseline_loss_player_0": float(self._last_baseline_loss[0]),
            "baseline_loss_player_1": float(self._last_baseline_loss[1]),
            "advantage_grad_norm_player_0": float(self._last_advantage_grad_norm[0]),
            "advantage_grad_norm_player_1": float(self._last_advantage_grad_norm[1]),
            "baseline_grad_norm_player_0": float(self._last_baseline_grad_norm[0]),
            "baseline_grad_norm_player_1": float(self._last_baseline_grad_norm[1]),
            "policy_grad_norm": float(self._last_policy_grad_norm),
            "strategy_buffer_size": int(len(self._strategy_memories)),
            "advantage_buffer_size_player_0": int(len(self._advantage_memories[0])),
            "advantage_buffer_size_player_1": int(len(self._advantage_memories[1])),
            "baseline_buffer_size_player_0": int(len(self._baseline_replays[0])),
            "baseline_buffer_size_player_1": int(len(self._baseline_replays[1])),
        }
        row.update(policy_diag)
        row.update(advantage_diag)
        row.update(baseline_diag)
        return row

    def _policy_network_diagnostics(self, max_states: int = 64) -> Dict[str, float]:
        if len(self._strategy_memories) == 0:
            return {"legal_action_mass_mean": np.nan, "legal_action_mass_min": np.nan, "policy_entropy_mean": np.nan}
        samples = self._strategy_memories.sample_up_to(max_states)
        masses = []
        entropies = []
        for s in samples:
            info = torch.from_numpy(np.asarray(s.info_state, dtype=np.float32)[None, :])
            with torch.no_grad():
                probs = self._policy_softmax(self._policy_network(info))[0].cpu().numpy()
            legal = np.where(np.asarray(s.strategy_action_probs) > 0)[0].tolist()
            if not legal:
                continue
            p_legal = np.asarray([probs[a] for a in legal], dtype=np.float64)
            mass = float(np.sum(p_legal))
            masses.append(mass)
            if mass > 0:
                p_legal = p_legal / mass
                entropies.append(float(-np.sum(p_legal * np.log(np.maximum(p_legal, 1e-12)))))
        return {
            "legal_action_mass_mean": safe_mean(masses),
            "legal_action_mass_min": float(np.min(masses)) if masses else float("nan"),
            "policy_entropy_mean": safe_mean(entropies),
        }

    def _advantage_target_summary(self, max_samples_per_player: int = 5000) -> Dict[str, float]:
        vals = []
        weights = []
        for p in range(self._num_players):
            for s in self._advantage_memories[p].sample_up_to(max_samples_per_player):
                vals.extend(list(s.advantage))
                weights.append(float(s.weight))
        arr = np.asarray(vals, dtype=np.float64)
        arr = arr[np.isfinite(arr)]
        return {
            "advantage_target_count_sampled": int(len(arr)),
            "advantage_target_mean": float(arr.mean()) if len(arr) else np.nan,
            "advantage_target_variance": float(arr.var()) if len(arr) else np.nan,
            "advantage_target_abs_mean": float(np.mean(np.abs(arr))) if len(arr) else np.nan,
            "importance_weight_mean": safe_mean(weights),
            "importance_weight_std": safe_std(weights),
        }

    def _baseline_replay_summary(self, max_samples_per_player: int = 5000) -> Dict[str, float]:
        rewards = []
        for p in range(self._num_players):
            for tr in self._baseline_replays[p].sample_up_to(max_samples_per_player):
                rewards.append(float(tr.reward))
        return {
            "baseline_reward_mean_sampled": safe_mean(rewards),
            "baseline_reward_variance_sampled": float(np.var(rewards)) if len(rewards) else np.nan,
        }



## Experimental configuration

The default configuration is intentionally bounded. DREAM includes baseline learning in addition to advantage and average-policy learning, so the default protocol uses five matched seeds and a 500-iteration training horizon. Optional ten-seed settings are provided for a stronger final thesis run if compute permits.


In [ ]:

EXPERIMENT_CONFIG = {
    "experiment_name": "kuhn_poker_dream_multiseed_baseline",
    "game_name": "kuhn_poker",
    "algorithm": "DREAM-style OpenSpiel baseline",
    "num_iterations": 500,
    "num_traversals": 320,
    "evaluation_interval": 25,
    "policy_network_train_every": 25,
    "policy_network_train_steps": 200,
    "advantage_network_train_steps": 100,
    "baseline_network_train_steps": 100,
    "policy_network_layers": [32, 32],
    "advantage_network_layers": [32, 32],
    "baseline_network_layers": [32, 32],
    "learning_rate": 0.003,
    "batch_size_advantage": 1024,
    "batch_size_strategy": 1024,
    "batch_size_baseline": 1024,
    "advantage_memory_capacity": int(1e6),
    "strategy_memory_capacity": int(1e6),
    "baseline_memory_capacity": int(1e6),
    "epsilon": 0.06,
    "compute_exploitability": True,
    "seeds": DEFAULT_SEEDS_5,
    "optional_thesis_seeds_10": THESIS_SEEDS_10,
    "kuhn_game_value_player_0": KUHN_GAME_VALUE_P0,
    "average_policy_value_target": KUHN_AVERAGE_POLICY_VALUE_TARGET,
    "exploitability_threshold": EXPLOITABILITY_THRESHOLD,
    "output_root": "outputs/kuhn_poker_dream_multiseed_baseline",
}

EXPERIMENT_CONFIG


In [ ]:

def run_single_seed(seed: int, config: Dict, output_dir: Path) -> Tuple[pd.DataFrame, Dict]:
    if pyspiel is None:
        raise RuntimeError("OpenSpiel is not installed. Install open_spiel before running this notebook.")
    set_seed(seed)
    game = pyspiel.load_game(config["game_name"])
    solver = DREAMSolver(
        game=game,
        policy_network_layers=config["policy_network_layers"],
        advantage_network_layers=config["advantage_network_layers"],
        baseline_network_layers=config["baseline_network_layers"],
        num_iterations=config["num_iterations"],
        num_traversals=config["num_traversals"],
        learning_rate=config["learning_rate"],
        batch_size_advantage=config["batch_size_advantage"],
        batch_size_strategy=config["batch_size_strategy"],
        batch_size_baseline=config["batch_size_baseline"],
        advantage_memory_capacity=config["advantage_memory_capacity"],
        strategy_memory_capacity=config["strategy_memory_capacity"],
        baseline_memory_capacity=config["baseline_memory_capacity"],
        epsilon=config["epsilon"],
        advantage_network_train_steps=config["advantage_network_train_steps"],
        policy_network_train_steps=config["policy_network_train_steps"],
        baseline_network_train_steps=config["baseline_network_train_steps"],
        policy_network_train_every=config["policy_network_train_every"],
        compute_exploitability=config["compute_exploitability"],
        seed=seed,
    )
    curves = solver.solve()
    curves.insert(0, "seed", int(seed))
    curves.insert(1, "variant", "dream_baseline")
    seed_dir = ensure_dir(output_dir / f"seed_{seed}")
    curves.to_csv(seed_dir / "checkpoint_curves.csv", index=False)

    final_row = curves.iloc[-1]
    final_window = curves.tail(min(5, len(curves)))
    reached = curves[curves["exploitability"] <= config["exploitability_threshold"]]
    summary = {
        "seed": int(seed),
        "variant": "dream_baseline",
        "final_iteration": int(final_row["iteration"]),
        "final_nodes_touched": int(final_row["nodes_touched"]),
        "final_wall_clock_seconds": float(final_row["wall_clock_seconds"]),
        "final_exploitability": float(final_row["exploitability"]),
        "best_exploitability": float(curves["exploitability"].min()),
        "final_window_mean_exploitability": float(final_window["exploitability"].mean()),
        "final_window_std_exploitability": float(final_window["exploitability"].std(ddof=1)) if len(final_window) > 1 else 0.0,
        "exploitability_auc_by_iteration": compute_auc(curves["iteration"], curves["exploitability"]),
        "exploitability_auc_by_nodes": compute_auc(curves["nodes_touched"], curves["exploitability"]),
        "final_policy_value_player_0": float(final_row["policy_value_player_0"]),
        "final_average_policy_value": float(final_row["average_policy_value"]),
        "final_window_mean_average_policy_value": float(final_window["average_policy_value"].mean()),
        "final_average_policy_value_error": float(final_row["average_policy_value_error"]),
        "final_policy_value_error": float(final_row["policy_value_error"]),
        "best_policy_value_error": float(curves["policy_value_error"].min()),
        "nodes_to_threshold": int(reached.iloc[0]["nodes_touched"]) if len(reached) else np.nan,
        "iterations_to_threshold": int(reached.iloc[0]["iteration"]) if len(reached) else np.nan,
        "seconds_to_threshold": float(reached.iloc[0]["wall_clock_seconds"]) if len(reached) else np.nan,
    }
    with open(seed_dir / "seed_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    return curves, summary


def run_experiment(config: Dict) -> Tuple[pd.DataFrame, pd.DataFrame, Path]:
    output_root = ensure_dir(config["output_root"])
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = ensure_dir(output_root / run_id)
    with open(output_dir / "experiment_metadata.json", "w") as f:
        json.dump(config, f, indent=2)

    all_curves = []
    summaries = []
    for seed in config["seeds"]:
        print(f"Running DREAM baseline seed {seed}...")
        curves, summary = run_single_seed(seed, config, output_dir)
        all_curves.append(curves)
        summaries.append(summary)

    curves_df = pd.concat(all_curves, ignore_index=True)
    summary_df = pd.DataFrame(summaries)
    curves_df.to_csv(output_dir / "checkpoint_curves.csv", index=False)
    summary_df.to_csv(output_dir / "seed_summary.csv", index=False)

    aggregate = {}
    for col in [
        "final_exploitability", "best_exploitability", "final_window_mean_exploitability",
        "exploitability_auc_by_iteration", "final_policy_value_error", "final_wall_clock_seconds",
        "final_nodes_touched",
    ]:
        aggregate[f"{col}_mean"] = safe_mean(summary_df[col])
        aggregate[f"{col}_se"] = standard_error(summary_df[col])
    with open(output_dir / "aggregate_summary.json", "w") as f:
        json.dump(aggregate, f, indent=2)

    return curves_df, summary_df, output_dir


In [ ]:

# Run the experiment.
# For a quick check, temporarily set EXPERIMENT_CONFIG["seeds"] = SMOKE_TEST_SEEDS
# and reduce num_iterations / num_traversals.

# curves_df, summary_df, output_dir = run_experiment(EXPERIMENT_CONFIG)
# summary_df


## Plotting helpers

In [ ]:

NASH_EXPLOITABILITY_TARGET = 0.0
NASH_EXPLOITABILITY_TARGET_LABEL = "Nash equilibrium target (0)"
AVERAGE_POLICY_VALUE_TARGET_LABEL = "Average-policy value target"


def is_exploitability_metric(metric: str) -> bool:
    return "exploitability" in metric.lower()


def is_average_policy_value_metric(metric: str) -> bool:
    metric = metric.lower()
    return metric.endswith("average_policy_value") and not metric.startswith("delta_")


def add_nash_exploitability_target(ax) -> None:
    ax.axhline(
        NASH_EXPLOITABILITY_TARGET,
        linestyle="--",
        linewidth=1.1,
        label=NASH_EXPLOITABILITY_TARGET_LABEL,
    )


def add_average_policy_value_target(ax, target: float = KUHN_AVERAGE_POLICY_VALUE_TARGET) -> None:
    ax.axhline(
        float(target),
        linestyle="--",
        linewidth=1.1,
        label=f"{AVERAGE_POLICY_VALUE_TARGET_LABEL} ({target:.4f})",
    )


def plot_mean_curve(
    curves_df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    ylabel: str,
    output_path: Optional[Path] = None,
    equilibrium_line: Optional[float] = None,
    average_policy_value_target: float = KUHN_AVERAGE_POLICY_VALUE_TARGET,
):
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    for seed, seed_df in curves_df.groupby("seed"):
        ax.plot(seed_df[x_col], seed_df[y_col], linewidth=0.9, alpha=0.28)
    grouped = curves_df.groupby(x_col)[y_col]
    mean = grouped.mean()
    se = grouped.sem().fillna(0.0)
    x = mean.index.to_numpy(dtype=float)
    y = mean.to_numpy(dtype=float)
    se_y = se.to_numpy(dtype=float)
    ax.plot(x, y, linewidth=2.2, label="Mean across seeds")
    ax.fill_between(x, y - se_y, y + se_y, alpha=0.18, label="±1 s.e.")
    if is_exploitability_metric(y_col):
        add_nash_exploitability_target(ax)
    elif is_average_policy_value_metric(y_col):
        add_average_policy_value_target(ax, average_policy_value_target)
    elif equilibrium_line is not None:
        ax.axhline(equilibrium_line, linestyle="--", linewidth=1.1, label="Target")
    ax.set_title(title)
    ax.set_xlabel(x_col.replace("_", " ").title())
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
    return fig, ax


def plot_summary_bar(summary_df: pd.DataFrame, metric: str, title: str, ylabel: str, output_path: Optional[Path] = None, average_policy_value_target: float = KUHN_AVERAGE_POLICY_VALUE_TARGET):
    fig, ax = plt.subplots(figsize=(6.8, 4.6))
    mean = summary_df[metric].mean()
    se = summary_df[metric].sem()
    ax.bar(["DREAM baseline"], [mean], yerr=[se], capsize=6)
    if is_exploitability_metric(metric):
        add_nash_exploitability_target(ax)
        ax.legend()
    elif is_average_policy_value_metric(metric):
        add_average_policy_value_target(ax, average_policy_value_target)
        ax.legend()
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
    return fig, ax


def create_thesis_plots(curves_df: pd.DataFrame, summary_df: pd.DataFrame, output_dir: Path) -> None:
    plot_dir = ensure_dir(output_dir / "plots")
    plot_mean_curve(
        curves_df,
        "iteration",
        "exploitability",
        "DREAM Baseline: Exploitability by Iteration",
        "Exploitability",
        plot_dir / "dream_exploitability_by_iteration.png",
    )
    plot_mean_curve(
        curves_df,
        "nodes_touched",
        "exploitability",
        "DREAM Baseline: Exploitability by Nodes Touched",
        "Exploitability",
        plot_dir / "dream_exploitability_by_nodes.png",
    )
    plot_mean_curve(
        curves_df,
        "iteration",
        "average_policy_value",
        "DREAM Baseline: Average Policy Value by Iteration",
        "Average policy value",
        plot_dir / "dream_average_policy_value_by_iteration.png",
        average_policy_value_target=EXPERIMENT_CONFIG["average_policy_value_target"],
    )
    plot_mean_curve(
        curves_df,
        "nodes_touched",
        "average_policy_value",
        "DREAM Baseline: Average Policy Value by Nodes Touched",
        "Average policy value",
        plot_dir / "dream_average_policy_value_by_nodes.png",
        average_policy_value_target=EXPERIMENT_CONFIG["average_policy_value_target"],
    )
    plot_mean_curve(
        curves_df,
        "iteration",
        "policy_value_error",
        "DREAM Baseline: Policy-Value Error",
        "Absolute error from -1/18",
        plot_dir / "dream_policy_value_error.png",
    )
    plot_mean_curve(
        curves_df,
        "iteration",
        "policy_loss",
        "DREAM Baseline: Average-Policy Loss Diagnostic",
        "Policy loss",
        plot_dir / "dream_policy_loss.png",
    )
    plot_mean_curve(
        curves_df,
        "iteration",
        "advantage_target_variance",
        "DREAM Baseline: Advantage-Target Variance Diagnostic",
        "Target variance",
        plot_dir / "dream_advantage_target_variance.png",
    )
    plot_mean_curve(
        curves_df,
        "iteration",
        "baseline_reward_variance_sampled",
        "DREAM Baseline: Baseline-Replay Reward Variance Diagnostic",
        "Reward variance",
        plot_dir / "dream_baseline_replay_variance.png",
    )
    plot_summary_bar(
        summary_df,
        "final_exploitability",
        "DREAM Baseline: Final Exploitability",
        "Final exploitability",
        plot_dir / "dream_final_exploitability.png",
    )
    plot_summary_bar(
        summary_df,
        "final_average_policy_value",
        "DREAM Baseline: Final Average Policy Value",
        "Final average policy value",
        plot_dir / "dream_final_average_policy_value.png",
        average_policy_value_target=EXPERIMENT_CONFIG["average_policy_value_target"],
    )

# After running the experiment, call:
# create_thesis_plots(curves_df, summary_df, output_dir)



## Suggested reporting table

After running the experiment, the following cell creates a compact table suitable for conversion into a thesis results table.


In [ ]:

# Example after running:
# report_cols = [
#     "final_exploitability",
#     "best_exploitability",
#     "final_window_mean_exploitability",
#     "exploitability_auc_by_iteration",
#     "final_policy_value_error",
#     "final_nodes_touched",
#     "final_wall_clock_seconds",
# ]
# thesis_table = pd.DataFrame({
#     "metric": report_cols,
#     "mean": [summary_df[c].mean() for c in report_cols],
#     "standard_error": [summary_df[c].sem() for c in report_cols],
# })
# thesis_table
